# Phase 3 · Day 15 · ComplaintClassificationProject

**Date:** 2026-04-24

Today you will build a small Turkish complaint classification project from end to end.

Learning objectives:
- Create a small text classification dataset inline.
- Clean Turkish complaint text with a simple function.
- Convert text into numbers with TF-IDF.
- Train and evaluate a classifier.
- Inspect mistakes and try a simple prediction function.


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

pd.set_option('display.max_colwidth', 120)
np.random.seed(42)
print('Setup complete')


In [ ]:
complaints = [
    ('Kartımdan iki kez para çekildi, iade istiyorum.', 'payment'),
    ('Ödeme yaptım ama sistem başarısız gösteriyor.', 'payment'),
    ('Kredi kartı ödemem hesaba düşmedi.', 'payment'),
    ('Fatura ödedim ama borç hala açık görünüyor.', 'payment'),
    ('Para transferim karşı hesaba geçmedi.', 'payment'),
    ('İade tutarı kartıma yansımadı.', 'payment'),
    ('Mobil uygulama açılmıyor, sürekli hata veriyor.', 'technical'),
    ('Şifre yenileme ekranı çalışmıyor.', 'technical'),
    ('Uygulamada giriş yaparken beyaz ekran geliyor.', 'technical'),
    ('Bildirimlere tıklayınca uygulama kapanıyor.', 'technical'),
    ('Web sitesinde sipariş sayfası yüklenmiyor.', 'technical'),
    ('SMS doğrulama kodu gelmiyor.', 'technical'),
    ('Kargo üç gündür dağıtımda görünüyor.', 'delivery'),
    ('Siparişim teslim edildi yazıyor ama bana ulaşmadı.', 'delivery'),
    ('Kurye adresimi bulamadığını söyledi.', 'delivery'),
    ('Paketim hasarlı geldi ve ürün kırılmış.', 'delivery'),
    ('Teslimat tarihi sürekli erteleniyor.', 'delivery'),
    ('Yanlış adrese teslimat yapılmış.', 'delivery'),
    ('Müşteri hizmetlerine bağlanamıyorum.', 'support'),
    ('Destek ekibi talebime cevap vermedi.', 'support'),
    ('Canlı destek konuşmayı yarıda kapattı.', 'support'),
    ('Çağrı merkezi beni sürekli aktarıyor.', 'support'),
    ('Şikayet kaydıma dönüş yapılmadı.', 'support'),
    ('Temsilci sorunu anlamadan görüşmeyi bitirdi.', 'support'),
    ('Hesabıma giriş yapamıyorum, kullanıcı bulunamadı diyor.', 'account'),
    ('E-posta adresimi değiştiremiyorum.', 'account'),
    ('Hesabım güvenlik nedeniyle kilitlendi.', 'account'),
    ('Profil bilgilerim yanlış görünüyor.', 'account'),
    ('Telefon numaramı güncellemek istiyorum ama olmuyor.', 'account'),
    ('Hesabımda eski adresim kayıtlı kalmış.', 'account'),
]

df = pd.DataFrame(complaints, columns=['text', 'label'])
print(df.shape)
df.sample(5, random_state=42)


## 1. Understand the dataset

A text classification project starts with a labeled dataset.

Here, each row has one complaint text and one category. The category is the target we want to predict.


In [ ]:
print('Label counts:')
print(df['label'].value_counts())

print('\nExample complaints:')
display(df.head(8))


In [ ]:
summary = (
    df.assign(text_length=df['text'].str.len(), word_count=df['text'].str.split().str.len())
      .groupby('label')[['text_length', 'word_count']]
      .mean()
      .round(1)
)
summary


## 2. Clean Turkish text

Text data is usually messy. We can lowercase the text and remove punctuation.

For Turkish text, this simple approach is not perfect, but it is a good first baseline.


In [ ]:
def clean_text(text):
    text = text.replace('I', 'ı').replace('İ', 'i').lower()
    text = re.sub(r'[^a-zA-ZçğıöşüÇĞİÖŞÜ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

examples = df['text'].head(5).tolist()
for original in examples:
    print('Original:', original)
    print('Cleaned :', clean_text(original))
    print()


In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text', 'label']].head()


## 3. Convert text into features with TF-IDF

A model cannot read raw text directly. TF-IDF converts text into numeric features.

Words that are useful for a category get higher weight. Very common words get lower weight.


In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_tfidf = vectorizer.fit_transform(df['clean_text'])

print('TF-IDF matrix shape:', X_tfidf.shape)
print('Number of features:', len(vectorizer.get_feature_names_out()))
print('Sample features:', vectorizer.get_feature_names_out()[:20])


In [ ]:
feature_names = vectorizer.get_feature_names_out()
first_row = X_tfidf[0].toarray().ravel()
non_zero = pd.DataFrame({
    'feature': feature_names[first_row > 0],
    'tfidf': first_row[first_row > 0]
}).sort_values('tfidf', ascending=False)

print(df.loc[0, 'clean_text'])
non_zero.head(10)


## 4. Train a baseline classifier

A baseline is the first simple model. It helps us understand what is already working.

We will use Logistic Regression because it works well for many text classification tasks.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.30,
    random_state=42,
    stratify=df['label']
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))
print('\nTrain labels:')
print(y_train.value_counts())


In [ ]:
model = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)
preds = model.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, preds), 3))
print('\nClassification report:')
print(classification_report(y_test, preds, zero_division=0))


## 5. Inspect mistakes

Accuracy is useful, but it is not enough.

Looking at wrong predictions helps you understand where the model is confused.


In [ ]:
results = pd.DataFrame({
    'text': X_test.values,
    'true_label': y_test.values,
    'predicted_label': preds
})
results['is_correct'] = results['true_label'] == results['predicted_label']

print('Prediction results:')
display(results)

print('Mistakes only:')
display(results[~results['is_correct']])


In [ ]:
labels = sorted(df['label'].unique())
cm = pd.DataFrame(confusion_matrix(y_test, preds, labels=labels), index=labels, columns=labels)
cm.index.name = 'true'
cm.columns.name = 'predicted'
cm


## 6. Predict new complaints

After training, we can wrap prediction in a small function.

This makes the model easier to use and test.


In [ ]:
def predict_complaint_category(text):
    cleaned = clean_text(text)
    prediction = model.predict([cleaned])[0]
    probabilities = model.predict_proba([cleaned])[0]
    classes = model.named_steps['clf'].classes_
    prob_table = pd.DataFrame({'label': classes, 'probability': probabilities}).sort_values('probability', ascending=False)
    return prediction, prob_table

new_text = 'Siparişim teslim edildi görünüyor ama paket bana gelmedi.'
prediction, prob_table = predict_complaint_category(new_text)
print('Text:', new_text)
print('Prediction:', prediction)
display(prob_table)


In [ ]:
test_texts = [
    'Kartımdan para çekildi ama sipariş oluşmadı.',
    'Uygulama giriş ekranında donuyor.',
    'Müşteri temsilcisi hiç yardımcı olmadı.',
    'Telefon numaramı hesabımda değiştiremiyorum.'
]

for text in test_texts:
    prediction, _ = predict_complaint_category(text)
    print(f'{prediction:10s} -> {text}')


## 7. Cross-validation

A single train-test split can be noisy, especially with small data.

Cross-validation trains and tests the model several times on different splits.


In [ ]:
cv_model = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

scores = cross_val_score(cv_model, df['clean_text'], df['label'], cv=3, scoring='accuracy')
print('CV scores:', np.round(scores, 3))
print('Mean CV accuracy:', round(scores.mean(), 3))


## Tricky bits

Small text datasets are fragile. A model may look good on one split and weaker on another.

Also, cleaning can remove useful information if you make it too aggressive.


In [ ]:
# Mistake 1: fitting TF-IDF separately on train and test creates different columns.
train_vectorizer = TfidfVectorizer()
test_vectorizer = TfidfVectorizer()

X_train_wrong = train_vectorizer.fit_transform(X_train)
X_test_wrong = test_vectorizer.fit_transform(X_test)

print('Train columns:', X_train_wrong.shape[1])
print('Test columns :', X_test_wrong.shape[1])

try:
    wrong_clf = LogisticRegression(max_iter=1000).fit(X_train_wrong, y_train)
    wrong_clf.predict(X_test_wrong)
except ValueError as e:
    print('What breaks:', e)


In [ ]:
# Mistake 2: using raw text directly in Logistic Regression.
try:
    LogisticRegression(max_iter=1000).fit(X_train.to_frame(), y_train)
except ValueError as e:
    print('What breaks:', e)


## Trick questions

1. Why do we use a Pipeline for text classification?
<details><summary>Answer</summary>
It keeps preprocessing and modeling together. This helps avoid train-test mismatch and makes prediction easier.
</details>

2. Why is accuracy risky on imbalanced data?
<details><summary>Answer</summary>
A model can predict the majority class often and still get high accuracy. Check precision, recall, F1, and confusion matrix.
</details>

3. Should we fit TF-IDF on the full dataset before train-test split?
<details><summary>Answer</summary>
No. That leaks information from the test set. Fit preprocessing only on training data.
</details>

4. Why might Turkish text need special care?
<details><summary>Answer</summary>
Turkish has special characters and rich suffixes. Simple cleaning is a baseline, not a full language solution.
</details>

5. What does ngram_range=(1, 2) mean?
<details><summary>Answer</summary>
The vectorizer uses single words and two-word phrases as features.
</details>


In [ ]:
# Exercise 1
# Create a copy of df called project_df.
project_df = df.copy()

assert isinstance(project_df, pd.DataFrame)
assert project_df.shape == df.shape
print('Exercise 1 passed')


In [ ]:
# Exercise 2
# Count how many complaints exist per label.
label_counts = df['label'].value_counts()

assert label_counts.loc['payment'] == 6
assert label_counts.loc['technical'] == 6
print('Exercise 2 passed')


In [ ]:
# Exercise 3
# Clean this text using clean_text.
raw_text = 'Uygulama AÇILMIYOR!!! SMS kodu da gelmiyor.'
cleaned_text = clean_text(raw_text)

assert cleaned_text == 'uygulama açılmıyor sms kodu da gelmiyor'
print('Exercise 3 passed')


In [ ]:
# Exercise 4
# Create a TF-IDF vectorizer with unigrams and bigrams.
my_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
my_matrix = my_vectorizer.fit_transform(df['clean_text'])

assert my_matrix.shape[0] == len(df)
assert 'müşteri' in my_vectorizer.get_feature_names_out()
print('Exercise 4 passed')


In [ ]:
# Exercise 5
# Build a pipeline with TF-IDF and Logistic Regression.
my_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
my_pipeline.fit(X_train, y_train)
my_preds = my_pipeline.predict(X_test)

assert len(my_preds) == len(y_test)
assert set(my_preds).issubset(set(df['label']))
print('Exercise 5 passed')


In [ ]:
# Exercise 6
# Predict the category for a new complaint.
complaint = 'Kargo teslim edildi yazıyor ama ürün gelmedi.'
my_prediction = model.predict([clean_text(complaint)])[0]

assert my_prediction in set(df['label'])
print('Exercise 6 passed:', my_prediction)


In [ ]:
# Exercise 7
# Create a DataFrame with true and predicted labels.
evaluation_df = pd.DataFrame({'true_label': y_test.values, 'predicted_label': preds})

assert list(evaluation_df.columns) == ['true_label', 'predicted_label']
assert len(evaluation_df) == len(y_test)
print('Exercise 7 passed')


## Solutions for today's exercises

<details><summary>Exercise 1</summary>

```python
project_df = df.copy()
```
</details>

<details><summary>Exercise 2</summary>

```python
label_counts = df['label'].value_counts()
```
</details>

<details><summary>Exercise 3</summary>

```python
cleaned_text = clean_text(raw_text)
```
</details>

<details><summary>Exercise 4</summary>

```python
my_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
```
</details>

<details><summary>Exercise 5</summary>

```python
my_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])
```
</details>

<details><summary>Exercise 6</summary>

```python
my_prediction = model.predict([clean_text(complaint)])[0]
```
</details>

<details><summary>Exercise 7</summary>

```python
evaluation_df = pd.DataFrame({'true_label': y_test.values, 'predicted_label': preds})
```
</details>


## Cumulative review exercises

These exercises mix skills from recent days: pandas, NumPy, splitting data, preprocessing, pipelines, metrics, and model interpretation.


In [ ]:
# Review 1: pandas filtering
# Select only delivery complaints.
delivery_df = df[df['label'] == 'delivery']

assert len(delivery_df) == 6
assert delivery_df['label'].eq('delivery').all()
print('Review 1 passed')


In [ ]:
# Review 2: feature engineering
# Add a word_count column to df.
df_review = df.copy()
df_review['word_count'] = df_review['text'].str.split().str.len()

assert 'word_count' in df_review.columns
assert df_review['word_count'].min() > 0
print('Review 2 passed')


In [ ]:
# Review 3: NumPy mean
# Calculate the average text length using NumPy.
length_array = df['text'].str.len().to_numpy()
avg_length = np.mean(length_array)

assert isinstance(avg_length, (float, np.floating))
assert avg_length > 20
print('Review 3 passed')


In [ ]:
# Review 4: train-test split
# Split clean_text and label with 25% test size.
X_tr, X_te, y_tr, y_te = train_test_split(df['clean_text'], df['label'], test_size=0.25, random_state=42, stratify=df['label'])

assert len(X_tr) + len(X_te) == len(df)
assert len(X_te) == 8
print('Review 4 passed')


In [ ]:
# Review 5: simple preprocessing check
# Clean every text and store it as a list.
cleaned_list = [clean_text(text) for text in df['text']]

assert isinstance(cleaned_list, list)
assert len(cleaned_list) == len(df)
assert cleaned_list[0] == df.loc[0, 'clean_text']
print('Review 5 passed')


In [ ]:
# Review 6: pipeline prediction
# Use the trained model to predict three texts.
mini_texts = ['Şifremi değiştiremiyorum.', 'Ödeme yaptım ama görünmüyor.', 'Paket yanlış adrese gitmiş.']
mini_preds = model.predict([clean_text(text) for text in mini_texts])

assert len(mini_preds) == 3
assert set(mini_preds).issubset(set(df['label']))
print('Review 6 passed:', mini_preds)


In [ ]:
# Review 7: metric calculation
# Calculate accuracy for y_test and preds.
acc = accuracy_score(y_test, preds)

assert 0 <= acc <= 1
print('Review 7 passed:', round(acc, 3))


In [ ]:
# Review 8: groupby aggregation
# Calculate average word count per label.
word_summary = df_review.groupby('label')[['word_count']].mean()

assert set(word_summary.index) == set(df['label'].unique())
assert 'word_count' in word_summary.columns
print('Review 8 passed')


In [ ]:
# Review 9: probability output
# Get prediction probabilities for one cleaned text.
one_text = clean_text('Uygulama sürekli hata veriyor.')
probabilities = model.predict_proba([one_text])

assert probabilities.shape[1] == len(model.named_steps['clf'].classes_)
assert np.isclose(probabilities.sum(), 1.0)
print('Review 9 passed')


In [ ]:
# Review 10: sort values
# Sort results so wrong predictions appear first.
sorted_results = results.sort_values('is_correct')

assert sorted_results.iloc[0]['is_correct'] in [False, True]
assert len(sorted_results) == len(results)
print('Review 10 passed')


## Cumulative review solutions

<details><summary>Review 1</summary>

```python
delivery_df = df[df['label'] == 'delivery']
```
</details>

<details><summary>Review 2</summary>

```python
df_review['word_count'] = df_review['text'].str.split().str.len()
```
</details>

<details><summary>Review 3</summary>

```python
avg_length = np.mean(length_array)
```
</details>

<details><summary>Review 4</summary>

```python
X_tr, X_te, y_tr, y_te = train_test_split(df['clean_text'], df['label'], test_size=0.25, random_state=42, stratify=df['label'])
```
</details>

<details><summary>Review 5</summary>

```python
cleaned_list = [clean_text(text) for text in df['text']]
```
</details>

<details><summary>Review 6</summary>

```python
mini_preds = model.predict([clean_text(text) for text in mini_texts])
```
</details>

<details><summary>Review 7</summary>

```python
acc = accuracy_score(y_test, preds)
```
</details>

<details><summary>Review 8</summary>

```python
word_summary = df_review.groupby('label')[['word_count']].mean()
```
</details>

<details><summary>Review 9</summary>

```python
probabilities = model.predict_proba([one_text])
```
</details>

<details><summary>Review 10</summary>

```python
sorted_results = results.sort_values('is_correct')
```
</details>


In [ ]:
cheat_sheet = '''
Complaint Classification Quick Reference

1. Create labels:
   df = pd.DataFrame({'text': texts, 'label': labels})

2. Clean text:
   text.lower()
   re.sub(...) for punctuation

3. Split data:
   train_test_split(X, y, test_size=0.3, stratify=y)

4. Build pipeline:
   Pipeline([
       ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
       ('clf', LogisticRegression(max_iter=1000))
   ])

5. Evaluate:
   accuracy_score(y_test, preds)
   classification_report(y_test, preds)
   confusion_matrix(y_test, preds)

6. Predict:
   model.predict([clean_text(new_complaint)])
'''
print(cheat_sheet)


## Next up: Day 16 — ModelEvaluationAndErrorAnalysis

You will go deeper into model evaluation, confusion matrices, error analysis, and better decision metrics.
